# Data Combination

This notebook converts the raw Zenodo wind farm CSV files into a **memory-safe interim layer** with **one Parquet file per turbine**.

## Goals
- discover wind farm and turbine files from the raw layer
- load and preprocess one turbine at a time
- write one Parquet file per turbine to the interim layer
- generate manifest and metadata summary artifacts
- preserve schema-awareness without forcing cross-farm feature alignment

## Important methodological note
Raw sensor names are **not guaranteed to be semantically consistent across farms**. This notebook therefore performs **structural preprocessing only** and does **not** attempt to harmonize features across farms. That mapping is deferred until `feature_description.csv` is used later in the pipeline.


## Setup Notebook

In [1]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
from IPython.display import display

from wtfd.config.config import (
    load_config, 
    get_path,
)
from wtfd.data_processing.discovery import (
    find_wind_farm_directories,
    find_turbine_csv_files,
    load_scada_csv,
)
from wtfd.data_processing.preprocessing import preprocess_scada_dataframe
from wtfd.data_processing.schema import (
    build_turbine_metadata_row,
    extract_column_presence_record,
)

## Configuration and Paths

In [3]:
config = load_config()

RAW_DATA_DIR = get_path("raw_zenodo", config)
INTERIM_DIR = get_path("turbine_parquet", config)
ARTIFACTS_DIR = get_path("data_combination_artifacts", config)

MANIFEST_FILENAME = "turbine_output_manifest.csv"
METADATA_FILENAME = "turbine_metadata_summary.csv"
COLUMN_PRESENCE_FILENAME = "column_presence_long.csv"
RUN_SUMMARY_FILENAME = "data_combination_run_summary.csv"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DATA_DIR :", RAW_DATA_DIR)
print("INTERIM_DIR  :", INTERIM_DIR)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)

[OK] Loaded configuration from wtfd.config.load_config()
PROJECT_ROOT : /mnt/c/grad_school/northeastern/courses/ie7275/project/wind-turbine-fault-detection
RAW_DATA_DIR : /mnt/c/grad_school/northeastern/courses/ie7275/project/wind-turbine-fault-detection/data/raw/zenodo_windfarm_data
INTERIM_DIR  : /mnt/c/grad_school/northeastern/courses/ie7275/project/wind-turbine-fault-detection/data/interim/turbine_parquet
ARTIFACTS_DIR: /mnt/c/grad_school/northeastern/courses/ie7275/project/wind-turbine-fault-detection/data/artifacts/data_combination


## Discover Wind Farms and Turbine CSV Files

In [4]:
wind_farm_dirs = find_wind_farm_directories(
    raw_data_dir=RAW_DATA_DIR,
    require_datasets_subdir=True,
)

farm_records = []
turbine_file_records = []

for wind_farm_dir in wind_farm_dirs:
    turbine_csvs = find_turbine_csv_files(wind_farm_dir)
    farm_records.append(
        {
            "wind_farm": wind_farm_dir.name,
            "wind_farm_dir": str(wind_farm_dir),
            "n_turbine_files": len(turbine_csvs),
        }
    )

    for csv_path in turbine_csvs:
        turbine_file_records.append(
            {
                "wind_farm": wind_farm_dir.name,
                "csv_path": str(csv_path),
                "csv_name": csv_path.name,
                "turbine_id": csv_path.stem,
            }
        )

wind_farm_summary_df = pd.DataFrame(farm_records).sort_values("wind_farm").reset_index(drop=True)
turbine_files_df = pd.DataFrame(turbine_file_records).sort_values(["wind_farm", "turbine_id"]).reset_index(drop=True)

display(wind_farm_summary_df)
display(turbine_files_df.head(10))

print(f"Total wind farms discovered: {len(wind_farm_summary_df)}")
print(f"Total turbine CSV files discovered: {len(turbine_files_df)}")

,wind_farm,wind_farm_dir,n_turbine_files
0,Wind Farm A,/mnt/c/grad_school/northeastern/courses/ie7275...,22
1,Wind Farm B,/mnt/c/grad_school/northeastern/courses/ie7275...,15
2,Wind Farm C,/mnt/c/grad_school/northeastern/courses/ie7275...,58


,wind_farm,csv_path,csv_name,turbine_id
0,Wind Farm A,/mnt/c/grad_school/northeastern/courses/ie7275...,0.csv,0
1,Wind Farm A,/mnt/c/grad_school/northeastern/courses/ie7275...,10.csv,10
2,Wind Farm A,/mnt/c/grad_school/northeastern/courses/ie7275...,13.csv,13
3,Wind Farm A,/mnt/c/grad_school/northeastern/courses/ie7275...,14.csv,14
4,Wind Farm A,/mnt/c/grad_school/northeastern/courses/ie7275...,17.csv,17
5,Wind Farm A,/mnt/c/grad_school/northeastern/courses/ie7275...,22.csv,22
6,Wind Farm A,/mnt/c/grad_school/northeastern/courses/ie7275...,24.csv,24
7,Wind Farm A,/mnt/c/grad_school/northeastern/courses/ie7275...,25.csv,25
8,Wind Farm A,/mnt/c/grad_school/northeastern/courses/ie7275...,26.csv,26
9,Wind Farm A,/mnt/c/grad_school/northeastern/courses/ie7275...,3.csv,3


Total wind farms discovered: 3
Total turbine CSV files discovered: 95


## Define Output Path Helpers

In [5]:
def sanitize_name(value: str) -> str:
    return (
        str(value)
        .strip()
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
    )


def build_turbine_parquet_path(interim_dir: Path, wind_farm: str, turbine_id: str) -> Path:
    farm_dir = interim_dir / sanitize_name(wind_farm)
    farm_dir.mkdir(parents=True, exist_ok=True)
    return farm_dir / f"{sanitize_name(turbine_id)}.parquet"

## Process Turbines 

In [6]:
metadata_rows = []
manifest_rows = []
column_presence_frames = []
error_rows = []

for idx, row in turbine_files_df.iterrows():
    wind_farm = row["wind_farm"]
    turbine_id = row["turbine_id"]
    csv_path = Path(row["csv_path"])
    output_path = build_turbine_parquet_path(
        interim_dir=INTERIM_DIR,
        wind_farm=wind_farm,
        turbine_id=turbine_id,
    )

    print(f"[{idx + 1}/{len(turbine_files_df)}] Processing {wind_farm} :: {turbine_id}")

    try:
        # Load one turbine file only
        raw_df = load_scada_csv(csv_path)

        # Structural preprocessing only
        turbine_df = preprocess_scada_dataframe(
            df=raw_df,
            wind_farm=wind_farm,
            turbine_id=turbine_id,
            source_file=csv_path.name,
        )

        # Write one parquet per turbine
        turbine_df.to_parquet(output_path, index=False)

        # Metadata summary row
        metadata_row = build_turbine_metadata_row(
            df=turbine_df,
            wind_farm=wind_farm,
            turbine_id=turbine_id,
            source_file=csv_path.name,
        )
        metadata_row["output_parquet_path"] = str(output_path)
        metadata_rows.append(metadata_row)

        # Manifest row
        manifest_rows.append(
            {
                "wind_farm": wind_farm,
                "turbine_id": turbine_id,
                "source_csv_path": str(csv_path),
                "source_csv_name": csv_path.name,
                "output_parquet_path": str(output_path),
                "n_rows_written": int(len(turbine_df)),
                "n_columns_written": int(turbine_df.shape[1]),
                "status": "success",
            }
        )

        # Long-format column presence artifact
        column_presence_df = extract_column_presence_record(
            df=turbine_df,
            wind_farm=wind_farm,
            turbine_id=turbine_id,
        )
        column_presence_frames.append(column_presence_df)

        # Free memory before next file
        del raw_df
        del turbine_df

    except Exception as exc:
        error_message = str(exc)
        print(f"[ERROR] {wind_farm} :: {turbine_id} -> {error_message}")

        manifest_rows.append(
            {
                "wind_farm": wind_farm,
                "turbine_id": turbine_id,
                "source_csv_path": str(csv_path),
                "source_csv_name": csv_path.name,
                "output_parquet_path": str(output_path),
                "n_rows_written": None,
                "n_columns_written": None,
                "status": "error",
            }
        )

        error_rows.append(
            {
                "wind_farm": wind_farm,
                "turbine_id": turbine_id,
                "source_csv_path": str(csv_path),
                "error_message": error_message,
            }
        )

[1/95] Processing Wind Farm A :: 0
[2/95] Processing Wind Farm A :: 10
[3/95] Processing Wind Farm A :: 13
[4/95] Processing Wind Farm A :: 14
[5/95] Processing Wind Farm A :: 17
[6/95] Processing Wind Farm A :: 22
[7/95] Processing Wind Farm A :: 24
[8/95] Processing Wind Farm A :: 25
[9/95] Processing Wind Farm A :: 26
[10/95] Processing Wind Farm A :: 3
[11/95] Processing Wind Farm A :: 38
[12/95] Processing Wind Farm A :: 40
[13/95] Processing Wind Farm A :: 42
[14/95] Processing Wind Farm A :: 45
[15/95] Processing Wind Farm A :: 51
[16/95] Processing Wind Farm A :: 68
[17/95] Processing Wind Farm A :: 69
[18/95] Processing Wind Farm A :: 71
[19/95] Processing Wind Farm A :: 72
[20/95] Processing Wind Farm A :: 73
[21/95] Processing Wind Farm A :: 84
[22/95] Processing Wind Farm A :: 92
[23/95] Processing Wind Farm B :: 19
[24/95] Processing Wind Farm B :: 2
[25/95] Processing Wind Farm B :: 21
[26/95] Processing Wind Farm B :: 23
[27/95] Processing Wind Farm B :: 27
[28/95] Proce

## Build Summary DataFrames

In [7]:
manifest_df = pd.DataFrame(manifest_rows).sort_values(["wind_farm", "turbine_id"]).reset_index(drop=True)
metadata_summary_df = pd.DataFrame(metadata_rows).sort_values(["wind_farm", "turbine_id"]).reset_index(drop=True)

if column_presence_frames:
    column_presence_long_df = (
        pd.concat(column_presence_frames, ignore_index=True)
        .sort_values(["wind_farm", "turbine_id", "column_name"])
        .reset_index(drop=True)
    )
else:
    column_presence_long_df = pd.DataFrame(
        columns=["wind_farm", "turbine_id", "column_name", "dtype"]
    )

errors_df = pd.DataFrame(error_rows)

run_summary_df = pd.DataFrame(
    [
        {
            "n_wind_farms": int(len(wind_farm_summary_df)),
            "n_turbine_files_discovered": int(len(turbine_files_df)),
            "n_successful_outputs": int((manifest_df["status"] == "success").sum()) if not manifest_df.empty else 0,
            "n_failed_outputs": int((manifest_df["status"] == "error").sum()) if not manifest_df.empty else 0,
            "interim_output_dir": str(INTERIM_DIR),
            "artifacts_dir": str(ARTIFACTS_DIR),
        }
    ]
)

display(run_summary_df)
display(manifest_df.head())
display(metadata_summary_df.head())
display(column_presence_long_df.head())
if not errors_df.empty:
    display(errors_df.head())

,n_wind_farms,n_turbine_files_discovered,n_successful_outputs,n_failed_outputs,interim_output_dir,artifacts_dir
0,3,95,95,0,/mnt/c/grad_school/northeastern/courses/ie7275...,/mnt/c/grad_school/northeastern/courses/ie7275...


,wind_farm,turbine_id,source_csv_path,source_csv_name,output_parquet_path,n_rows_written,n_columns_written,status
0,Wind Farm A,0,/mnt/c/grad_school/northeastern/courses/ie7275...,0.csv,/mnt/c/grad_school/northeastern/courses/ie7275...,54986,90,success
1,Wind Farm A,10,/mnt/c/grad_school/northeastern/courses/ie7275...,10.csv,/mnt/c/grad_school/northeastern/courses/ie7275...,53592,90,success
2,Wind Farm A,13,/mnt/c/grad_school/northeastern/courses/ie7275...,13.csv,/mnt/c/grad_school/northeastern/courses/ie7275...,54010,90,success
3,Wind Farm A,14,/mnt/c/grad_school/northeastern/courses/ie7275...,14.csv,/mnt/c/grad_school/northeastern/courses/ie7275...,54197,90,success
4,Wind Farm A,17,/mnt/c/grad_school/northeastern/courses/ie7275...,17.csv,/mnt/c/grad_school/northeastern/courses/ie7275...,55090,90,success


,wind_farm,turbine_id,source_file,n_rows,n_columns,has_timestamp,timestamp_min,timestamp_max,n_missing_timestamps,column_names,output_parquet_path
0,Wind Farm A,0,0.csv,54986,90,True,2022-08-04 06:10:00,2023-08-24 06:10:00,0,wind_farm|turbine_id|source_file|timestamp|tim...,/mnt/c/grad_school/northeastern/courses/ie7275...
1,Wind Farm A,10,10.csv,53592,90,True,2022-10-09 08:40:00,2023-10-18 08:40:00,0,wind_farm|turbine_id|source_file|timestamp|tim...,/mnt/c/grad_school/northeastern/courses/ie7275...
2,Wind Farm A,13,13.csv,54010,90,True,2022-04-30 13:20:00,2023-05-25 10:20:00,0,wind_farm|turbine_id|source_file|timestamp|tim...,/mnt/c/grad_school/northeastern/courses/ie7275...
3,Wind Farm A,14,14.csv,54197,90,True,2022-03-03 14:00:00,2023-03-16 18:40:00,0,wind_farm|turbine_id|source_file|timestamp|tim...,/mnt/c/grad_school/northeastern/courses/ie7275...
4,Wind Farm A,17,17.csv,55090,90,True,2022-10-31 15:20:00,2023-11-20 00:40:00,0,wind_farm|turbine_id|source_file|timestamp|tim...,/mnt/c/grad_school/northeastern/courses/ie7275...


,wind_farm,turbine_id,column_name,dtype
0,Wind Farm A,0,asset_id,int64
1,Wind Farm A,0,id,int64
2,Wind Farm A,0,power_29_avg,float64
3,Wind Farm A,0,power_29_max,float64
4,Wind Farm A,0,power_29_min,float64


## Save Artifacts

In [8]:
manifest_path = ARTIFACTS_DIR / MANIFEST_FILENAME
metadata_path = ARTIFACTS_DIR / METADATA_FILENAME
column_presence_path = ARTIFACTS_DIR / COLUMN_PRESENCE_FILENAME
run_summary_path = ARTIFACTS_DIR / RUN_SUMMARY_FILENAME
errors_path = ARTIFACTS_DIR / "data_combination_errors.csv"
wind_farm_summary_path = ARTIFACTS_DIR / "wind_farm_summary.csv"
turbine_files_path = ARTIFACTS_DIR / "turbine_files_discovered.csv"

manifest_df.to_csv(manifest_path, index=False)
metadata_summary_df.to_csv(metadata_path, index=False)
column_presence_long_df.to_csv(column_presence_path, index=False)
run_summary_df.to_csv(run_summary_path, index=False)
wind_farm_summary_df.to_csv(wind_farm_summary_path, index=False)
turbine_files_df.to_csv(turbine_files_path, index=False)

if not errors_df.empty:
    errors_df.to_csv(errors_path, index=False)

print(f"Saved manifest: {manifest_path}")
print(f"Saved metadata summary: {metadata_path}")
print(f"Saved column presence: {column_presence_path}")
print(f"Saved run summary: {run_summary_path}")
print(f"Saved wind farm summary: {wind_farm_summary_path}")
print(f"Saved turbine file inventory: {turbine_files_path}")
if not errors_df.empty:
    print(f"Saved errors: {errors_path}")

Saved manifest: /mnt/c/grad_school/northeastern/courses/ie7275/project/wind-turbine-fault-detection/data/artifacts/data_combination/turbine_output_manifest.csv
Saved metadata summary: /mnt/c/grad_school/northeastern/courses/ie7275/project/wind-turbine-fault-detection/data/artifacts/data_combination/turbine_metadata_summary.csv
Saved column presence: /mnt/c/grad_school/northeastern/courses/ie7275/project/wind-turbine-fault-detection/data/artifacts/data_combination/column_presence_long.csv
Saved run summary: /mnt/c/grad_school/northeastern/courses/ie7275/project/wind-turbine-fault-detection/data/artifacts/data_combination/data_combination_run_summary.csv
Saved wind farm summary: /mnt/c/grad_school/northeastern/courses/ie7275/project/wind-turbine-fault-detection/data/artifacts/data_combination/wind_farm_summary.csv
Saved turbine file inventory: /mnt/c/grad_school/northeastern/courses/ie7275/project/wind-turbine-fault-detection/data/artifacts/data_combination/turbine_files_discovered.csv


## Quick Validation Checks

In [9]:

successful_manifest_df = manifest_df.loc[manifest_df["status"] == "success"].copy()

print("Successful outputs:", len(successful_manifest_df))
print("Failed outputs    :", len(errors_df))

if not successful_manifest_df.empty:
    sample_output_path = Path(successful_manifest_df.iloc[0]["output_parquet_path"])
    print("Sample parquet file:", sample_output_path)

    sample_parquet_df = pd.read_parquet(sample_output_path)
    display(sample_parquet_df.head())

    expected_front_cols = [col for col in ["wind_farm", "turbine_id", "source_file", "timestamp"] if col in sample_parquet_df.columns]
    print("Leading standardized columns:", expected_front_cols)


Successful outputs: 95
Failed outputs    : 0
Sample parquet file: /mnt/c/grad_school/northeastern/courses/ie7275/project/wind-turbine-fault-detection/data/interim/turbine_parquet/Wind_Farm_A/0.parquet


,wind_farm,turbine_id,source_file,timestamp,time_stamp,asset_id,id,train_test,status_type_id,sensor_0_avg,...,sensor_47,sensor_48,sensor_49,sensor_50,sensor_51,sensor_52_avg,sensor_52_max,sensor_52_min,sensor_52_std,sensor_53_avg
0,Wind Farm A,0,0.csv,2022-08-04 06:10:00,2022-08-04 06:10:00,0,0,train,0,22.0,...,-496.0,0.0,0.0,-1280.0,-496.0,0.0,0.0,0.0,0.0,26.0
1,Wind Farm A,0,0.csv,2022-08-04 06:20:00,2022-08-04 06:20:00,0,1,train,0,22.0,...,-490.0,0.0,0.0,-1278.0,-490.0,0.0,0.0,0.0,0.0,25.0
2,Wind Farm A,0,0.csv,2022-08-04 06:30:00,2022-08-04 06:30:00,0,2,train,0,22.0,...,-490.0,0.0,0.0,-1356.0,-490.0,0.0,0.0,0.0,0.0,25.0
3,Wind Farm A,0,0.csv,2022-08-04 06:40:00,2022-08-04 06:40:00,0,3,train,0,22.0,...,-509.0,0.0,0.0,-1274.0,-509.0,0.0,0.0,0.0,0.0,26.0
4,Wind Farm A,0,0.csv,2022-08-04 06:50:00,2022-08-04 06:50:00,0,4,train,0,22.0,...,-499.0,0.0,0.0,-1284.0,-499.0,0.0,0.0,0.0,0.0,26.0


Leading standardized columns: ['wind_farm', 'turbine_id', 'source_file', 'timestamp']


## Interpretation

At the end of this notebook, the project has a stable **interim layer** with one Parquet file per turbine.

This supports the next stages of the pipeline because:
- raw CSV parsing does not need to be repeated in later notebooks
- turbine-level processing remains memory-safe
- metadata and schema artifacts are preserved for auditability
- feature harmonization is still deferred until metadata-driven mapping is available

The next notebook should read from the interim Parquet layer rather than from the raw CSV files.